# ✅ Solutions — Session 4: GroupBy & Merging

Worked answers to every exercise plus the **Orders + Customers** mini project.
Each solution is self-contained and uses pandas 3.x idioms only.

In [1]:
import numpy as np
import pandas as pd

## Exercise 1 — Total revenue per region (named aggregation)

**(Easy, coding)** Build the `sales` DataFrame, then group by `region` and sum `revenue`.

In [2]:
sales = pd.DataFrame({
    "region":  ["North", "South", "North", "South", "North", "South"],
    "product": ["Laptop", "Phone", "Phone", "Laptop", "Tablet", "Tablet"],
    "revenue": [1200, 800, 400, 1500, 500, 900],
})

sales.groupby("region").agg(total=("revenue", "sum"))

,total
region,
North,2100
South,3200


## Exercise 2 — Product counts per region (`crosstab`)

In [3]:
pd.crosstab(sales["region"], sales["product"], margins=True)

product,Laptop,Phone,Tablet,All
region,,,,
North,1,1,1,3
South,1,1,1,3
All,2,2,2,6


## Exercise 3 — Four statistics in one `.agg()` call

Compute `sum`, `mean`, `max`, and `count` together, then keep regions whose total
is above the average region total.

In [4]:
stats = sales.groupby("region").agg(
    total=("revenue", "sum"),
    avg=("revenue", "mean"),
    maximum=("revenue", "max"),
    orders=("revenue", "count"),
)
stats

,total,avg,maximum,orders
region,,,,
North,2100,700.000000,1200,3
South,3200,1066.666667,1500,3


In [5]:
above_average = stats[stats["total"] > stats["total"].mean()]
above_average

,total,avg,maximum,orders
region,,,,
South,3200,1066.666667,1500,3


## Exercise 4 — Group share with `transform`

In [6]:
sales = sales.assign(
    region_total=lambda d: d.groupby("region")["revenue"].transform("sum"),
)
sales["share_of_region"] = (sales["revenue"] / sales["region_total"]).round(3)
sales

,region,product,revenue,region_total,share_of_region
0,North,Laptop,1200,2100,0.571
1,South,Phone,800,3200,0.250
2,North,Phone,400,2100,0.190
3,South,Laptop,1500,3200,0.469
4,North,Tablet,500,2100,0.238
5,South,Tablet,900,3200,0.281


## Exercise 5 — Conceptual: `agg` vs `transform`, and join semantics

**`.agg()` vs `.transform()` on a `groupby`:**

- `.agg()` **reduces** each group to a single summary row. The result has one row per
  group, so it is *shorter* than the original data. Use it for reports and totals.
- `.transform()` returns a result with the **same index and length as the input**, so a
  group-level value can be attached back onto every original row. Use it for features
  such as share-of-group or group z-scores.

**Row counts after a merge:**

- `how="inner"` keeps only keys that appear in **both** frames. Unmatched rows from
  either side are silently dropped, so the result has *fewer* (or equal) rows than the
  inputs. If a key appears multiple times on both sides, matching rows *multiply*.
- `how="outer"` keeps **every** key from both frames, filling the missing side with
  `NaN`. It never silently drops a key, so it produces the largest result.

The code below demonstrates both behaviours.

In [7]:
left = pd.DataFrame({"id": [1, 2, 3], "left_val": ["a", "b", "c"]})
right = pd.DataFrame({"id": [2, 3, 4], "right_val": ["x", "y", "z"]})

print("inner row count:", len(left.merge(right, on="id", how="inner")))
print("outer row count:", len(left.merge(right, on="id", how="outer")))

left.merge(right, on="id", how="outer")

inner row count: 2
outer row count: 4


,id,left_val,right_val
0,1,a,NaN
1,2,b,x
2,3,c,y
3,4,NaN,z


## 🚀 Mini Project — Orders + Customers

### Step 1 — Build the `orders` table with a `revenue` column

In [8]:
orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "customer_id": [101, 102, 101, 103, 104, 102, 105, 103],
    "product": ["Laptop", "Phone", "Tablet", "Laptop", "Phone", "Laptop", "Tablet", "Tablet"],
    "quantity": [1, 2, 1, 1, 3, 1, 2, 1],
    "unit_price": [1200, 800, 500, 1200, 800, 1200, 500, 500],
})
orders = orders.assign(revenue=orders["quantity"] * orders["unit_price"])
orders

,order_id,customer_id,product,quantity,unit_price,revenue
0,1,101,Laptop,1,1200,1200
1,2,102,Phone,2,800,1600
2,3,101,Tablet,1,500,500
3,4,103,Laptop,1,1200,1200
4,5,104,Phone,3,800,2400
5,6,102,Laptop,1,1200,1200
6,7,105,Tablet,2,500,1000
7,8,103,Tablet,1,500,500


### Step 2 — Build the `customers` table (Frank has no orders)

In [9]:
customers = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 105, 106],
    "name": ["Alice", "Bob", "Carol", "Dan", "Eve", "Frank"],
    "region": ["North", "South", "North", "East", "West", "South"],
})
customers

,customer_id,name,region
0,101,Alice,North
1,102,Bob,South
2,103,Carol,North
3,104,Dan,East
4,105,Eve,West
5,106,Frank,South


### Step 3 — Left-merge so every order is kept

In [10]:
merged = orders.merge(customers, on="customer_id", how="left")
merged

,order_id,customer_id,product,quantity,unit_price,revenue,name,region
0,1,101,Laptop,1,1200,1200,Alice,North
1,2,102,Phone,2,800,1600,Bob,South
2,3,101,Tablet,1,500,500,Alice,North
3,4,103,Laptop,1,1200,1200,Carol,North
4,5,104,Phone,3,800,2400,Dan,East
5,6,102,Laptop,1,1200,1200,Bob,South
6,7,105,Tablet,2,500,1000,Eve,West
7,8,103,Tablet,1,500,500,Carol,North


### Step 4 — Summarize each region with named aggregation

In [11]:
by_region = merged.groupby("region").agg(
    total_revenue=("revenue", "sum"),
    orders=("order_id", "count"),
    avg_order=("revenue", "mean"),
)
by_region

,total_revenue,orders,avg_order
region,,,
East,2400,1,2400.0
North,3400,4,850.0
South,2800,2,1400.0
West,1000,1,1000.0


### Step 5 — Revenue by region and product (`pivot_table`)

In [12]:
merged.pivot_table(
    values="revenue",
    index="region",
    columns="product",
    aggfunc="sum",
    fill_value=0,
    margins=True,
)

product,Laptop,Phone,Tablet,All
region,,,,
East,0,2400,0,2400
North,2400,0,1000,3400
South,1200,1600,0,2800
West,0,0,1000,1000
All,3600,4000,2000,9600


### Step 6 (Bonus) — Outer merge reveals the customer with no orders

`how="outer"` keeps Frank (customer 106). Because Frank has no orders, his order
columns are `NaN`. The `how="left"` result above contained no such row.

In [13]:
merged_outer = orders.merge(customers, on="customer_id", how="outer")
merged_outer[merged_outer["order_id"].isna()]

,order_id,customer_id,product,quantity,unit_price,revenue,name,region
8,NaN,106,NaN,NaN,NaN,NaN,Frank,South
